<h1 style="padding-top: 25px; padding-bottom: 25px; text-align: left; padding-left: 10px; background-color: #F7F1E8; border-bottom: 4px solid #D52B1E; color: #1E1C24;"> <img style="float: left; padding-right: 10px;" src="https://raw.githubusercontent.com/pavlosprotopapas/elelem-chile-2026/main/assets/epoch_tour_mark.png" height="50px"> <a href="https://pavlosprotopapas.github.io/elelem-chile-2026/" target="_blank"><strong><font color="#1E1C24">ELELEM · Epoch Tour Chile 2026 — From LLMs to Agentic Systems</font></strong></a></h1>

# **<font color="#D52B1E">Agents II lab — Travel Planner</font>**

**ELELEM Chile 2026** · Santiago, 22–24 October 2026<br/>
**Day 2**<br/>
**Instructors:** Pavlos Protopapas (Harvard University) · Guillermo Cabrera-Vives (Universidad de Concepción)<br/>

<hr style="height:2pt">

## 📝 Make a Copy to Edit

This notebook is **view-only**. To edit it, follow these steps:

1. Click **File** > **Save a copy in Drive**.
2. Your own editable copy will open in a new tab.

Now you can modify and run the code freely!

# **Working with Agents**


<center><img src="https://drive.google.com/uc?export=view&id=1oCldVicFepwdYbPHcD8Ui4VeyQVJYgut" height="400"><center>



In this notebook, we will build an LLM agent from scratch. We will build a LLM based Travel Agent using mock tools to make searching hotels, flights, restaurants simpler. In a real LLM based Travel Planner you would connect to real APIs for Hotels, Airlines, Restaurant searches.

# **Learning Objectives**

By the end of this tutorial/guided demo, you will be able to:

- **Understand AI Agents:**
  - Define agents and their role in workflows.
  - Explain agentic workflows for complex tasks.

- **Set Up Mock Tools:**
  - Set up mock tools for City, Hotel, Flight, Restaurant, Attractions search

- **Implement LLM AI Agents:**
  - Implement AI agent for Travel planning.
  - Integrate Agents with the mock tools.

- **Analyze Applications:**
  - Review and analyze real-world travel planning needs.


# **Understanding Agents**

**What are Agents?**<br>
---
Agent is a system where an LLM, given instructions and tools, runs autonomously until a task is done.

An agentic workflow is any multi-step process that iteratively instructs large language models to complete complex tasks. <br>

So, we can think of agents here as a part of an agentic workflow that completes a particular task and hands over the result to the next agent, all of them in tandem utilising LLMs to complete the larger task given as a prompt.

# Prerequisites

Let's make sure we have everything needed. We'll need to have an OpenAI API key for some parts of this tutorial.

Before we can start using the [OpenAI API](https://openai.com/blog/openai-api), we'll need to sign up for an API key from OpenAI. We can do this by visiting the [OpenAI API Keys](https://platform.openai.com/api-keys) page and creating a new API key.

In [ ]:
# @title Install OpenAI Agents SDK
%pip install -q "openai-agents==0.3.2"

In [ ]:
# @title Imports
import asyncio
import re
import json
import threading
from datetime import date, datetime
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal

from agents import Agent, Runner, RunContextWrapper, SQLiteSession, function_tool

In [ ]:
# @title Setup OpenAI Key
# Using Google Colab's Secrets feature 🔑
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
import os
os.environ['OPENAI_API_KEY'] = api_key

# **Demo Travel Catalog**

In this section we will define some mock data for this demo

In [ ]:
# @title Catalog Data & Utils

#########################
# Data
#########################
CITIES = {
    "Seattle": {"airport": "SEA", "description": "Waterfront, markets, and museums."},
    "San Francisco": {"airport": "SFO", "description": "Bay views, parks, and neighborhoods."},
    "Los Angeles": {"airport": "LAX", "description": "Museums, beaches, and film attractions."},
}
HOTELS = {
    "SFO-BAY": ("San Francisco", "Bay Area Suite", 25000, True),
    "SFO-GATE": ("San Francisco", "Golden Gate Inn", 18000, False),
    "LAX-DOWNTOWN": ("Los Angeles", "LA Downtown Hotel", 22000, True),
    "LAX-HOLLYWOOD": ("Los Angeles", "Hollywood Stars Hotel", 17000, False),
    "SEA-HARBOR": ("Seattle", "Harbor Lodge", 20000, True),
    "SEA-CENTER": ("Seattle", "Seattle Downtown Hotel", 15000, False),
}
RESTAURANTS = [
    {"city": city, "name": f"{city} {cuisine} Kitchen", "cuisine": cuisine,
     "estimated_meal_usd": cost}
    for city in CITIES
    for cuisine, cost in [("Italian", 30), ("Mexican", 20), ("French", 45)]
]
ATTRACTIONS = {
    "SFO-BRIDGE": ("San Francisco", "Golden Gate Bridge", 0),
    "SFO-MUSEUM": ("San Francisco", "Bay Art Museum", 3000),
    "LAX-OBSERVATORY": ("Los Angeles", "Griffith Observatory grounds", 0),
    "LAX-MUSEUM": ("Los Angeles", "City Film Museum", 2500),
    "SEA-MARKET": ("Seattle", "Pike Place Market", 0),
    "SEA-MUSEUM": ("Seattle", "City Music Museum", 3000),
}

#########################
# Data Contracts
#########################
@dataclass
class TravelContext:
    # Agent's Travel context state
    checked_plan: dict | None = None
    budget_report: dict | None = None
    pending_proposal: dict | None = None
    approved_proposal: dict | None = None
    validated_this_turn: bool = False
    finalized_this_turn: bool = False

class PlannerReply(BaseModel):
    """The model must distinguish a final itinerary from a continuing turn."""
    kind: Literal["clarification", "options", "itinerary", "out_of_scope"]
    message: str

#########################
# Utils
#########################
OUTPUT_LOCK = threading.Lock()
MONEY_IN_PROSE = re.compile(r"\$\s*\d|\b\d+(?:\.\d+)?\s*(?:USD|dollars?)\b", re.IGNORECASE)

def canonical_city(value: str) -> str:
    for city in CITIES:
        if city.casefold() == value.strip().casefold():
            return city
    raise ValueError(f"Unsupported city {value!r}. Choose from {list(CITIES)}.")

def parse_date(value: str) -> date:
    return date.fromisoformat(value)

def flight_options(origin: str, destination: str, departure_date: str) -> list[dict]:
    origin, destination = canonical_city(origin), canonical_city(destination)
    if origin == destination:
        raise ValueError("Origin and destination must differ.")
    day = parse_date(departure_date)
    route = f"{CITIES[origin]['airport']}-{CITIES[destination]['airport']}"
    base = 15000 if {origin, destination} == {"San Francisco", "Los Angeles"} else 22000
    return [
        {"id": f"{route}-{day:%Y%m%d}-{i}", "origin": origin,
         "destination": destination, "date": day.isoformat(),
         "departure_local": hour, "airline": airline, "seats_available": 8,
         "price_basis": "per person, one way, economy, all-in mock fare",
         "price_cents": base + extra, "price_usd": (base + extra) / 100}
        for i, (hour, airline, extra) in enumerate([
            ("09:00", "Demo Air", 0), ("15:00", "Tutorial Airways", 5000)
        ], start=1)
    ]

def resolve_flight(flight_id: str) -> dict:
    parts = flight_id.split("-")
    if len(parts) != 4:
        raise ValueError("Use an exact flight ID returned by flight_search.")
    airports = {v["airport"]: k for k, v in CITIES.items()}
    origin, destination, raw_day, _ = parts
    if origin not in airports or destination not in airports:
        raise ValueError("Unknown airport in flight ID.")
    day = datetime.strptime(raw_day, "%Y%m%d").date().isoformat()
    for flight in flight_options(airports[origin], airports[destination], day):
        if flight["id"] == flight_id:
            return flight
    raise ValueError("Flight ID was not found in the mock catalog.")

def log_tool(name: str, details: str) -> None:
    with OUTPUT_LOCK:
        print(f"  [tool] {name}: {details}", flush=True)

def log_catalog(name: str, result: object) -> None:
    """Display canonical tool data instead of asking the model to repeat prices."""
    block = f"  [catalog] {name}:\n{json.dumps(result, indent=2)}"
    with OUTPUT_LOCK:
        print(block, flush=True)


# **Define Tools**

In this section we will create mock tools that will be used by the AI Agent for Travel planning

In [ ]:
# @title Search Tools

#########################
# Tools
#########################
def city_search(city: str = "") -> dict:
    """List supported cities, or get information about one city; empty lists all."""
    log_tool("city_search", city or "all cities")
    if not city:
        result = {"cities": CITIES, "note": "Only these three cities are supported."}
    else:
        name = canonical_city(city)
        result = {"city": name, **CITIES[name]}
    log_catalog("city_search", result)
    return result

def flight_search(origin: str, destination: str, departure_date: str) -> list[dict]:
    """Search fictional one-way economy flights; date must be YYYY-MM-DD."""
    log_tool("flight_search", f"{origin} -> {destination}, {departure_date}")
    result = flight_options(origin, destination, departure_date)
    log_catalog("flight_search", result)
    return result

def hotel_search(city: str, check_in: str, check_out: str,
                 pet_friendly: bool = False) -> list[dict]:
    """Search one-room hotel stays, including exact cost for the requested nights."""
    name = canonical_city(city)
    nights = (parse_date(check_out) - parse_date(check_in)).days
    if nights <= 0:
        raise ValueError("Check-out must be after check-in.")
    log_tool("hotel_search", f"{name}, {nights} nights, pet-friendly={pet_friendly}")
    result = [{"id": key, "city": c, "name": title, "pet_friendly": pets,
               "nightly_usd": cents / 100, "nights": nights,
               "stay_total_usd": cents * nights / 100, "room_capacity": 2}
              for key, (c, title, cents, pets) in HOTELS.items()
              if c == name and (not pet_friendly or pets)]
    log_catalog("hotel_search", result)
    return result

def restaurant_search(city: str, cuisine: str = "") -> list[dict]:
    """Find fictional restaurants; optionally filter by Italian, Mexican, or French."""
    name = canonical_city(city)
    log_tool("restaurant_search", f"{name}, {cuisine or 'all cuisines'}")
    result = [r for r in RESTAURANTS if r["city"] == name
              and (not cuisine or r["cuisine"].casefold() == cuisine.strip().casefold())]
    log_catalog("restaurant_search", result)
    return result

def attraction_search(city: str) -> list[dict]:
    """List fictional attractions and admission prices per person per visit."""
    name = canonical_city(city)
    log_tool("attraction_search", name)
    result = [{"id": key, "name": title, "admission_usd": cents / 100}
              for key, (c, title, cents) in ATTRACTIONS.items() if c == name]
    log_catalog("attraction_search", result)
    return result

In [ ]:
# @title Finalize Plan Tool

def finalize_plan(travel_context: TravelContext, outbound_id: str,
                       return_id: str, hotel_id: str, travelers: int,
                       budget_usd: int, daily_food_usd: int,
                       local_transport_usd: int, pet_friendly: bool,
                       attraction_ids: list[str]) -> dict:
    """Check the chosen round trip and budget before presenting a final itinerary.

    Use the user's budget and traveler count, and explicitly agreed allowances.
    Food is USD per person per calendar day; transport is USD for the whole trip.
    Pass daily_food_usd exactly as the per-person amount the user stated. For
    example, 50 per person for two travelers means daily_food_usd=50, never 100.
    Python multiplies by travelers and days; do not multiply tool arguments.
    Flights, hotel, and attractions must use IDs returned by the search tools.
    This checks prices and compatibility; it does not book anything.
    """
    log_tool("finalize_plan", "validating selections and calculating costs")
    result = check_itinerary(outbound_id, return_id, hotel_id, travelers, budget_usd,
                             daily_food_usd, local_transport_usd, pet_friendly, attraction_ids)
    proposal = {
        "outbound_id": outbound_id,
        "return_id": return_id,
        "hotel_id": hotel_id,
        "travelers": travelers,
        "budget_usd": budget_usd,
        "daily_food_usd": daily_food_usd,
        "local_transport_usd": local_transport_usd,
        "pet_friendly": pet_friendly,
        "attraction_ids": sorted(attraction_ids),
    }

    # A model tool call can propose a plan, but only a later human turn can
    # approve that exact proposal. This prevents the model from silently
    # inventing requirements or treating its own suggestion as confirmation.
    if proposal != travel_context.approved_proposal:
        travel_context.pending_proposal = proposal
        return {
            "status": "needs_confirmation",
            "requirements": proposal,
            "message": "Stop and wait for the user's next chat reply.",
        }

    travel_context.pending_proposal = None
    travel_context.budget_report = result
    travel_context.validated_this_turn = True
    if result["status"] == "within_budget":
        travel_context.checked_plan = result
        travel_context.finalized_this_turn = True
    return result

In [ ]:
# Test

# Attractions
result = attraction_search(city='Los Angeles')
print("Attractions")
display(result)

# **Business Logic**

In this section we will define some determinisitic checks and guardrails to help AI Agent for Travel planning

In [ ]:
# @title Business Logic

#########################
# Business Logic
#########################
def check_itinerary(outbound_id: str, return_id: str, hotel_id: str,
                    travelers: int, budget_usd: int, daily_food_usd: int,
                    local_transport_usd: int, pet_friendly: bool,
                    attraction_ids: list[str]) -> dict:
    """Validate a round trip and compute its budget from canonical catalog prices.

    This demo supports 1–2 travelers sharing one room. Food is an allowance
    per person per calendar day; transport is an allowance for the entire trip.
    Mock flight/hotel prices are all-in; no real-world fees are implied.
    """
    if travelers not in (1, 2):
        raise ValueError("This demo supports 1–2 travelers sharing one room.")
    if budget_usd <= 0 or daily_food_usd < 0 or local_transport_usd < 0:
        raise ValueError("Budget must be positive and allowances cannot be negative.")
    outbound, inbound = resolve_flight(outbound_id), resolve_flight(return_id)
    if (outbound["origin"], outbound["destination"]) != (inbound["destination"], inbound["origin"]):
        raise ValueError("Return flight must reverse the outbound route.")
    nights = (parse_date(inbound["date"]) - parse_date(outbound["date"])).days
    if nights <= 0:
        raise ValueError("Return date must be after the outbound date.")
    if hotel_id not in HOTELS:
        raise ValueError("Unknown hotel ID; use hotel_search.")
    city, hotel_name, nightly, pets = HOTELS[hotel_id]
    if city != outbound["destination"] or (pet_friendly and not pets):
        raise ValueError("Hotel does not meet the destination/pet-friendly requirements.")
    if len(set(attraction_ids)) != len(attraction_ids):
        raise ValueError("Choose each attraction at most once.")
    admission = 0
    for key in attraction_ids:
        if key not in ATTRACTIONS or ATTRACTIONS[key][0] != city:
            raise ValueError(f"Attraction {key!r} is not in the destination city.")
        admission += ATTRACTIONS[key][2] * travelers
    costs = {"round_trip_flights": (outbound["price_cents"] + inbound["price_cents"]) * travelers,
             "hotel": nightly * nights, "food_allowance": daily_food_usd * 100 * travelers * (nights + 1),
             "local_transport_allowance": local_transport_usd * 100, "attractions": admission}
    total = sum(costs.values())
    return {"status": "within_budget" if total <= budget_usd * 100 else "over_budget",
            "booking_status": "Not booked — fictional planning only",
            "origin": outbound["origin"], "destination": city,
            "start_date": outbound["date"], "end_date": inbound["date"],
            "travelers": travelers, "rooms": 1, "nights": nights,
            "pet_friendly_required": pet_friendly,
            "outbound_flight": outbound_id, "return_flight": return_id,
            "hotel": hotel_name, "attractions": [ATTRACTIONS[k][1] for k in attraction_ids],
            "costs_usd": {k: v / 100 for k, v in costs.items()},
            "total_usd": total / 100, "budget_usd": budget_usd,
            "remaining_usd": (budget_usd * 100 - total) / 100,
            "assumptions": "One shared room; food for every calendar day, including arrival/departure; transport is a whole-trip allowance."}

def apply_confirmation(prompt: str, context: TravelContext) -> None:
    """Let only a later, explicit human response approve a pending proposal."""
    if context.pending_proposal is None:
        return
    answer = prompt.strip().casefold().rstrip(".! ")
    if answer in {"y", "yes", "confirm", "confirmed", "yes, confirm", "yes please"}:
        context.approved_proposal = context.pending_proposal.copy()
    else:
        # Corrections and unrelated replies never approve the previous proposal.
        context.approved_proposal = None
    context.pending_proposal = None

def checked_itinerary_reply(plan: dict) -> PlannerReply:
    """Use canonical checked data for the final prose, without repeating prices."""
    attractions = ", ".join(plan["attractions"]) or "no paid attractions"
    return PlannerReply(
        kind="itinerary",
        message=(
            f"Your checked round trip from {plan['origin']} to {plan['destination']} "
            f"is ready for {plan['travelers']} traveler(s), with {plan['hotel']} and "
            f"{attractions}. Budget report is shown below. Nothing has been booked."
        ),
    )

def confirmation_reply(context: TravelContext) -> PlannerReply:
    """Render the exact Python-owned proposal that needs human approval."""
    return PlannerReply(
        kind="clarification",
        message=(
            "Please review this exact proposed plan and reply with 'yes' to approve it, "
            "or tell me what to change. Food is per person per calendar day; local "
            "transport is for the whole trip.\n"
            + json.dumps(context.pending_proposal, indent=2)
        ),
    )

async def run_turn(agent, prompt,session, context, invoke_agent) -> PlannerReply:
    """
    Run one conversation turn and enforce confirmation, validation, and presentation.
    The model chooses actions and produces replies, while Python decides whether the resulting state permits those replies.
    """

    # Input validations
    if not prompt.strip():
        raise ValueError("The prompt cannot be empty.")

    # State Tracking
    context.validated_this_turn = False
    context.finalized_this_turn = False

    # Human approval and state tracking
    apply_confirmation(prompt, context)

    # Run Agent
    reply = await invoke_agent(agent, prompt, session=session, context=context)

    # Postcondition checking and corrective retry
    if reply.kind == "itinerary" and not context.finalized_this_turn and context.pending_proposal is None:
        reply = await invoke_agent(
            agent,
            "Application validation: an itinerary is not final until you call "
            "finalize_itinerary in this turn. Call it now with the exact selected "
            "plan. If requirements or approval are missing, return clarification "
            "or options instead.",
            session=session, context=context
        )

    # Authoritative presentation: determines what the user actually sees from the checked state
    if context.pending_proposal is not None:
        # Human confirmation
        reply = confirmation_reply(context)
    elif context.finalized_this_turn:
        # Render the validated final itinerary
        reply = checked_itinerary_reply(context.checked_plan)
    elif reply.kind == "itinerary":
        # Hard Guardrail: an unchecked final itinerary is rejected.
        raise RuntimeError("The agent returned an itinerary without a successful same-turn check.")

    # Soft Guardrail: it detects certain monetary expressions but still displays the response.
    if MONEY_IN_PROSE.search(reply.message):
        print("  [warning] The model repeated a monetary amount in narrative output.")

    print(f"Kind: {reply.kind}")
    print(reply.message)

    # Budget Report
    if context.validated_this_turn and context.budget_report is not None:
        print("Budget report (computed by Python; fictional planning only):")
        print(json.dumps(context.budget_report, indent=2))

    return reply


# **Setup Agent**

In this section we setup all the components we need to run an AI Agent for Travel planning.

The agent will have six tools. Its structured reply distinguishes clarification, options, a checked itinerary, and out-of-scope requests. The model summarizes; Python owns the arithmetic.

In [ ]:
# @title Travel Agent Instructions

#########################
# Instructions
#########################
TRAVEL_AGENT_INSTRUCTIONS = """You help plan fictional round trips for a classroom demo.
Only Seattle, San Francisco, and Los Angeles are supported, for 1–2 travelers
sharing one hotel room. All tools use fictional data. Never claim to book,
reserve, pay, or contact anyone. Politely decline unrelated requests.
Ask concise clarifying questions when the destination, dates, traveler count,
or total USD budget is missing. Never silently replace the user's requirements.
For an ambiguous region such as California, ask which city they want.
Use tools for all prices and availability. Preserve dates, selections, and
preferences across turns. Search both outbound and return flights. If a user
asks for unsupported accommodation, explain the one-room hotel limitation.
Flight search prices are PER PERSON. Python displays catalog results and checked
budgets directly. Your message must be a SHORT conversational summary
with NO monetary amounts, currency symbols, or cost breakdowns. You may refer
to the displayed catalog or budget report. Never compute trip totals yourself.
Ask the user to provide separate food and local-transport allowances; never
invent them or combine their units. Food covers every calendar day per person,
while local transport is one whole-trip allowance.
If they change a preference, update the affected choices and recheck the budget.
Call finalize_itinerary before calling a plan final. If it reports over_budget,
explain the shortfall and propose cheaper choices; never silently raise budget.
The first call with a new proposal returns needs_confirmation. Stop immediately,
return a clarification, and wait for a later human reply. Only the application
can approve that exact proposal. After the human confirms, call
finalize_itinerary again in the new turn.
If validation fails, correct the inputs using the tools. Present search options
with their exact IDs so users can select them. A successful finalization is
displayed by the CLI as a canonical checked cost table; summarize it concisely.
Use reply kind 'itinerary' for every final or updated final plan. Always call
finalize_itinerary in THAT turn, even when only restaurant preferences changed.
For itinerary replies, omit the cost table from your message: Python prints it.
The checker verifies catalog prices and the supplied constraints, not real-world
availability or schedule feasibility. Do not overstate what it verifies.
Reply kinds:
- clarification: essential trip requirements are missing, or an exact
  proposal needs confirmation.
- options: present available flights, hotels, or attractions for selection.
  Use options even when asking the user to choose among them.
- itinerary: a final plan successfully checked in this turn.
- out_of_scope: the request is outside this tutorial's supported scope.
"""


In [ ]:
# @title Travel Agent

@function_tool(description_override=finalize_plan.__doc__)
def finalize_itinerary(travel_context: RunContextWrapper[TravelContext], outbound_id: str,
                       return_id: str, hotel_id: str, travelers: int,
                       budget_usd: int, daily_food_usd: int,
                       local_transport_usd: int, pet_friendly: bool,
                       attraction_ids: list[str]) -> dict:
    # Finalize plan
    return finalize_plan(travel_context.context, outbound_id, return_id, hotel_id, travelers, budget_usd, daily_food_usd, local_transport_usd, pet_friendly, attraction_ids)

async def invoke_agent(agent: Agent, prompt: str, session: SQLiteSession, context: TravelContext) -> PlannerReply:
    """Run one agent invocation and obtain its Pydantic response value."""

    # Run agent
    result = await Runner.run(
        agent, prompt, session=session, context=context, max_turns=12
    )
    if isinstance(result.final_output, PlannerReply):
        return result.final_output
    raise RuntimeError("Agent returned a response that did not match PlannerReply.")

def build_agent():
    # LLM
    MODEL_NAME = "gpt-5.6-luna"

    # Tools
    tools = [
        function_tool(city_search),
        function_tool(flight_search),
        function_tool(hotel_search),
        function_tool(restaurant_search),
        function_tool(attraction_search),finalize_itinerary
    ]

    # Session & Context
    session = SQLiteSession("travel-planner")
    context = TravelContext()

    # Agent
    agent = Agent(
        name="Travel Planner",
        model=MODEL_NAME,
        instructions=TRAVEL_AGENT_INSTRUCTIONS,
        tools=tools,
        output_type=PlannerReply,
    )

    return agent, session, context

# **Travel Planner Agent in Action**

## Example 1:

User Query:

 *Plan a round trip from Seattle to California from November 6 to 10, 2026. There are two adults sharing one room, our total budget is $6000, and the hotel must be pet-friendly.*




In [ ]:
# Build the Agent
agent, session, context = build_agent()

# Turn 1: core requirements, with one deliberate city ambiguity.
reply = await run_turn(agent, """
Plan a round trip from Seattle to California from November 6 to 10, 2026.
There are two adults sharing one room, our total budget is $6000, and the
hotel must be pet-friendly.
""", session, context, invoke_agent)
assert reply.kind == "clarification"

Kind: clarification
Which supported California city would you like: San Francisco or Los Angeles? Please also provide your separate food allowance per person per calendar day and local-transport allowance for the whole trip.


In [ ]:
# Turn 2: resolve the city and state both allowances with exact units.
reply = await run_turn(agent, """
San Francisco. Food allowance of $60 per person per day
and $150 total for local transportation for the whole trip.
""", session, context, invoke_agent)
assert reply.kind == "options"

  [tool] flight_search: Seattle -> San Francisco, 2026-11-06
  [catalog] flight_search:
[
  {
    "id": "SEA-SFO-20261106-1",
    "origin": "Seattle",
    "destination": "San Francisco",
    "date": "2026-11-06",
    "departure_local": "09:00",
    "airline": "Demo Air",
    "seats_available": 8,
    "price_basis": "per person, one way, economy, all-in mock fare",
    "price_cents": 22000,
    "price_usd": 220.0
  },
  {
    "id": "SEA-SFO-20261106-2",
    "origin": "Seattle",
    "destination": "San Francisco",
    "date": "2026-11-06",
    "departure_local": "15:00",
    "airline": "Tutorial Airways",
    "seats_available": 8,
    "price_basis": "per person, one way, economy, all-in mock fare",
    "price_cents": 27000,
    "price_usd": 270.0
  }
]
  [tool] flight_search: San Francisco -> Seattle, 2026-11-10
  [catalog] flight_search:
[
  {
    "id": "SFO-SEA-20261110-1",
    "origin": "San Francisco",
    "destination": "Seattle",
    "date": "2026-11-10",
    "departure_local": "09

In [ ]:
# Turn 3: select every catalog item and ask for the exact proposal.
reply = await run_turn(agent, """
I choose outbound flight SEA-SFO-20261106-1, return flight
SFO-SEA-20261110-1, hotel SFO-BAY, and attractions SFO-BRIDGE and
SFO-MUSEUM.
""", session, context, invoke_agent)
assert reply.kind == "clarification"
assert context.pending_proposal is not None

  [tool] finalize_plan: validating selections and calculating costs
Kind: clarification
Please review this exact proposed plan and reply with 'yes' to approve it, or tell me what to change. Food is per person per calendar day; local transport is for the whole trip.
{
  "outbound_id": "SEA-SFO-20261106-1",
  "return_id": "SFO-SEA-20261110-1",
  "hotel_id": "SFO-BAY",
  "travelers": 2,
  "budget_usd": 6000,
  "daily_food_usd": 60,
  "local_transport_usd": 150,
  "pet_friendly": true,
  "attraction_ids": [
    "SFO-BRIDGE",
    "SFO-MUSEUM"
  ]
}


In [ ]:
# Turn 4: only this later human turn approves the displayed proposal.
reply = await run_turn(agent, "yes", session, context, invoke_agent)
assert reply.kind == "itinerary"
assert context.finalized_this_turn
assert context.checked_plan is not None
assert context.checked_plan["total_usd"] == 2690.0

  [tool] finalize_plan: validating selections and calculating costs
Kind: itinerary
Your checked round trip from Seattle to San Francisco is ready for 2 traveler(s), with Bay Area Suite and Golden Gate Bridge, Bay Art Museum. Budget report is shown below. Nothing has been booked.
Budget report (computed by Python; fictional planning only):
{
  "status": "within_budget",
  "booking_status": "Not booked \u2014 fictional planning only",
  "origin": "Seattle",
  "destination": "San Francisco",
  "start_date": "2026-11-06",
  "end_date": "2026-11-10",
  "travelers": 2,
  "rooms": 1,
  "nights": 4,
  "pet_friendly_required": true,
  "outbound_flight": "SEA-SFO-20261106-1",
  "return_flight": "SFO-SEA-20261110-1",
  "hotel": "Bay Area Suite",
  "attractions": [
    "Golden Gate Bridge",
    "Bay Art Museum"
  ],
  "costs_usd": {
    "round_trip_flights": 880.0,
    "hotel": 1000.0,
    "food_allowance": 600.0,
    "local_transport_allowance": 150.0,
    "attractions": 60.0
  },
  "total_usd":

In [ ]:
# @title Start an interactive conversation
async def read_interactive_message() -> str:
    """Read one possibly multiline message, terminated by blank line or /send."""
    lines: list[str] = []
    while True:
        label = "You: " if not lines else "...  "
        try:
            line = await asyncio.to_thread(input, label)
        except EOFError:
            if lines:
                return "\n".join(lines).strip()
            raise

        command = line.strip().casefold()
        if not lines and command in {"exit", "quit"}:
            return command
        if command == "/send" or not line.strip():
            if lines:
                return "\n".join(lines).strip()
            continue
        lines.append(line)

async def chat_loop():
    # Build the Agent
    agent, session, context = build_agent()

    print("Travel Planner (interactive). Enter one or more lines, then use a blank")
    print("line or /send to submit. Commands: 'exit' or 'quit' to leave.\n")

    try:
        while True:
            try:
                # Read input off the event loop so we don't block it. Buffering
                # lines prevents a pasted paragraph from becoming several turns.
                prompt = await read_interactive_message()
            except (EOFError, KeyboardInterrupt):
                print("\nGoodbye!")
                break

            if not prompt:
                continue
            if prompt.casefold() in {"exit", "quit"}:
                print("Goodbye!")
                break

            try:
                await run_turn(agent, prompt, session, context, invoke_agent)
            except Exception as e:
                # Keep the session alive on a bad turn instead of crashing the demo.
                print(f"[error] {type(e).__name__}: {e}\n")
    finally:
        session.close()

In [ ]:
# Run the Agent
await chat_loop()

Travel Planner (interactive). Enter one or more lines, then use a blank
line or /send to submit. Commands: 'exit' or 'quit' to leave.

